# Fabric BCDR demo seed notebook

Creates a synthetic retail-style star schema in the primary lakehouse so the DR flow has realistic tables and data to recover.

In [ ]:
from datetime import date, timedelta
import random
from pyspark.sql.types import DateType, DoubleType, IntegerType, StringType, StructField, StructType

random.seed(42)

LAKEHOUSE_NAME = '__LAKEHOUSE_NAME__'
CUSTOMER_COUNT = 500
PRODUCT_COUNT = 24
REGION_COUNT = 8
FACT_ROW_COUNT = 5000

print(f'Seeding lakehouse: {LAKEHOUSE_NAME}')

In [ ]:
customer_schema = StructType([
    StructField('customer_id', IntegerType()),
    StructField('customer_segment', StringType()),
    StructField('customer_name', StringType()),
    StructField('state_code', StringType())
])

segments = ['Consumer', 'Corporate', 'Public Sector', 'Healthcare']
states = ['CA', 'TX', 'NY', 'FL', 'IL', 'WA', 'GA', 'NC']

customer_rows = [
    (idx, segments[idx % len(segments)], f'Customer_{idx:04d}', states[idx % len(states)])
    for idx in range(1, CUSTOMER_COUNT + 1)
]

product_schema = StructType([
    StructField('product_id', IntegerType()),
    StructField('product_category', StringType()),
    StructField('product_name', StringType()),
    StructField('list_price', DoubleType())
])

categories = ['Compute', 'Storage', 'Analytics', 'IoT', 'Security', 'AI']
product_rows = [
    (idx, categories[idx % len(categories)], f'Product_{idx:02d}', round(99 + (idx * 12.5), 2))
    for idx in range(1, PRODUCT_COUNT + 1)
]

region_schema = StructType([
    StructField('region_id', IntegerType()),
    StructField('region_name', StringType()),
    StructField('dr_pattern', StringType())
])

region_rows = [
    (1, 'East US', 'paired'),
    (2, 'West US 2', 'paired'),
    (3, 'Central US', 'nonpaired'),
    (4, 'North Europe', 'paired'),
    (5, 'West Europe', 'paired'),
    (6, 'Australia East', 'nonpaired'),
    (7, 'Japan East', 'nonpaired'),
    (8, 'UK South', 'paired')
]

calendar_schema = StructType([
    StructField('date_key', IntegerType()),
    StructField('calendar_date', DateType()),
    StructField('calendar_month', IntegerType()),
    StructField('calendar_quarter', IntegerType()),
    StructField('calendar_year', IntegerType())
])

start_date = date(2025, 1, 1)
calendar_rows = []
for offset in range(0, 365):
    current = start_date + timedelta(days=offset)
    calendar_rows.append((
        int(current.strftime('%Y%m%d')),
        current,
        current.month,
        ((current.month - 1) // 3) + 1,
        current.year
    ))

spark.createDataFrame(customer_rows, customer_schema).write.mode('overwrite').format('delta').saveAsTable(f'{LAKEHOUSE_NAME}.dim_customer')
spark.createDataFrame(product_rows, product_schema).write.mode('overwrite').format('delta').saveAsTable(f'{LAKEHOUSE_NAME}.dim_product')
spark.createDataFrame(region_rows, region_schema).write.mode('overwrite').format('delta').saveAsTable(f'{LAKEHOUSE_NAME}.dim_region')
spark.createDataFrame(calendar_rows, calendar_schema).write.mode('overwrite').format('delta').saveAsTable(f'{LAKEHOUSE_NAME}.dim_calendar')

print('Dimension tables ready')

In [ ]:
fact_schema = StructType([
    StructField('sale_id', IntegerType()),
    StructField('date_key', IntegerType()),
    StructField('customer_id', IntegerType()),
    StructField('product_id', IntegerType()),
    StructField('region_id', IntegerType()),
    StructField('units', IntegerType()),
    StructField('gross_sales', DoubleType()),
    StructField('support_tier', StringType())
])

support_tiers = ['Standard', 'Premium', 'MissionCritical']
date_keys = [row[0] for row in calendar_rows]
fact_rows = []

for sale_id in range(1, FACT_ROW_COUNT + 1):
    units = random.randint(1, 20)
    price = round(random.uniform(120.0, 4800.0), 2)
    fact_rows.append((
        sale_id,
        random.choice(date_keys),
        random.randint(1, CUSTOMER_COUNT),
        random.randint(1, PRODUCT_COUNT),
        random.randint(1, REGION_COUNT),
        units,
        round(units * price, 2),
        random.choice(support_tiers)
    ))

spark.createDataFrame(fact_rows, fact_schema).write.mode('overwrite').format('delta').saveAsTable(f'{LAKEHOUSE_NAME}.fact_sales')
print(f'fact_sales ready: {FACT_ROW_COUNT:,} rows')

In [ ]:
summary_tables = ['dim_customer', 'dim_product', 'dim_region', 'dim_calendar', 'fact_sales']
total_rows = 0
for table_name in summary_tables:
    row_count = spark.table(f'{LAKEHOUSE_NAME}.{table_name}').count()
    total_rows += row_count
    print(f'{table_name}: {row_count:,} rows')

print(f'Total rows written: {total_rows:,}')
print('Primary lakehouse sample data ready')